# Signal Playground

Purpose: create a deterministic synthetic CSI frame, inspect amplitude and phase, and optionally wrap it in the current `ruview.core.CsiFrame` contract.

Run path: install research extras with `uv sync --extra research`, open this notebook, and choose Run All. The cells are deterministic and do not need radio hardware.

Fixture / simulated source: `make_playground_frame` is an inline NumPy generator. If `ruview.core` imports successfully, the generated array is also wrapped as a `CsiFrame` to show frame shape and witness hash.


In [ ]:
import numpy as np

try:
    import matplotlib.pyplot as plt
except Exception as exc:
    raise RuntimeError('Install the research extra with: uv sync --extra research') from exc

try:
    from ruview.core import CsiFrame, CsiMetadata, FrequencyBand, Timestamp
except Exception as exc:
    CsiFrame = CsiMetadata = FrequencyBand = Timestamp = None
    CORE_IMPORT_ERROR = exc
else:
    CORE_IMPORT_ERROR = None


def make_playground_frame(streams=3, subcarriers=56, seed=17):
    rng = np.random.default_rng(seed)
    subcarrier_axis = np.linspace(-1.0, 1.0, subcarriers)
    data = np.empty((streams, subcarriers), dtype=np.complex128)

    for stream in range(streams):
        baseline = 1.0 + 0.08 * np.sin(2.0 * np.pi * (stream + 1) * subcarrier_axis)
        ripple = 0.03 * np.cos(8.0 * np.pi * subcarrier_axis + stream)
        amplitude = baseline + ripple + rng.normal(0.0, 0.006, subcarriers)
        phase = 0.25 * np.sin(np.pi * subcarrier_axis * (stream + 1)) + 0.04 * stream
        phase = phase + rng.normal(0.0, 0.01, subcarriers)
        data[stream] = amplitude * np.exp(1j * phase)

    return data


csi = make_playground_frame()
amplitude = np.abs(csi)
phase = np.unwrap(np.angle(csi), axis=1)
subcarrier_index = np.arange(csi.shape[1])

summary = {
    'shape': csi.shape,
    'source': 'inline synthetic fixture',
    'mean_amplitude': round(float(amplitude.mean()), 4),
    'amplitude_variance': round(float(amplitude.var()), 6),
    'mean_unwrapped_phase': round(float(phase.mean()), 4),
}

if CsiFrame is not None:
    metadata = CsiMetadata(
        'synthetic-playground',
        FrequencyBand.BAND_2_4_GHZ,
        6,
        timestamp=Timestamp(1_700_000_000, 0),
    )
    frame = CsiFrame(metadata, csi)
    summary['core_frame_streams'] = frame.num_spatial_streams()
    summary['core_frame_subcarriers'] = frame.num_subcarriers()
    summary['witness_hash'] = frame.witness_hash().hex()
else:
    summary['core_fallback'] = repr(CORE_IMPORT_ERROR)

summary


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)

for stream_index in range(csi.shape[0]):
    label = f'stream {stream_index}'
    axes[0].plot(subcarrier_index, amplitude[stream_index], label=label)
    axes[1].plot(subcarrier_index, phase[stream_index], label=label)

axes[0].set_title('Amplitude by stream')
axes[0].set_xlabel('Subcarrier index')
axes[0].set_ylabel('Amplitude')
axes[0].grid(True, alpha=0.3)
axes[0].legend(loc='best')

axes[1].set_title('Unwrapped phase by stream')
axes[1].set_xlabel('Subcarrier index')
axes[1].set_ylabel('Phase (radians)')
axes[1].grid(True, alpha=0.3)
axes[1].legend(loc='best')

plt.show()


Expected interpretation: smooth amplitude ripples and slowly varying phase indicate a stable synthetic channel. Different streams should have related but offset structure, which makes the plots useful for checking downstream stream/subcarrier handling.

Limitations: this is not calibrated RF data, does not model packet loss or carrier-frequency offset, and uses a simple phase unwrap. Treat the witness hash as a contract smoke check, not as evidence of physical realism.
